#### Import Necessary Packages

In [ ]:
from pyspark.sql.functions import current_timestamp, date_format
from pyspark.sql.types import StructType, StructField, StringType
from delta.tables import DeltaTable
import pandas as pd
import numpy as np
import re
import traceback

# Disable Arrow optimization for PySpark
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

StatementMeta(, fbd9461f-47e2-4653-9407-cf8584579aee, 9, Finished, Available, Finished)

#### Define Required Paths

In [ ]:
# Define the path to the configuration file in OneLake
lh_utils_path = 'abfss://450414b4-7ed1-4e2e-9325-41ea8d377b0b@onelake.dfs.fabric.microsoft.com/8db8a7bf-1457-44cf-89a7-df04f9eeaec0'
config_relative_path = '/Files/config_files/250722 DQ Config.xlsx'
config_path = f'{lh_utils_path}{config_relative_path}'

StatementMeta(, fbd9461f-47e2-4653-9407-cf8584579aee, 10, Finished, Available, Finished)

#### Excel-to-Lakehouse Data Loader: Dynamic Multi-Sheet Ingestion

In [9]:
# ====================================================================================================================
# `Function`: load_into_lakehouse
#
# `Inputs`: 
# - master_source_config: Pandas DataFrame containing metadata for each sheet to be loaded
# - config_path: Path to the source Excel file containing multiple sheets
# - lh_utils_path: Base Lakehouse path where Delta tables will be stored
#
# `Purpose`: Dynamically loads multiple Excel sheets into Delta Lake tables, converting all values to strings
#
# `Key Operations`:
# 1. Cleans configuration and sheet data, converting all values to strings and handling NaNs as NULLs
# 2. Adds audit timestamp and writes each sheet as a Delta table with schema overwrite support
#
# `Output`: Delta tables stored at path: {lh_utils_path}/Tables/{target_schema_name}/{target_table_name}
# ====================================================================================================================

def load_into_lakehouse(master_source_config):

    # Clean master config by removing any unnamed columns
    master_source_config = master_source_config.loc[:, ~master_source_config.columns.str.contains('^Unnamed')]
    
    # Extract configuration lists from master
    sheet_names = master_source_config['sheet_name'].tolist()
    target_schema_names = master_source_config['target_schema_name'].tolist()
    target_table_names = master_source_config['target_table_name'].tolist()

    # Process each sheet specified in the master configuration
    for sheet_name, target_schema_name, target_table_name in zip(sheet_names, target_schema_names, target_table_names):
        try:
            # Read and clean sheet data
            source_config_instance = pd.read_excel(config_path, sheet_name=sheet_name, dtype=object)
            source_config_instance = source_config_instance.loc[:, ~source_config_instance.columns.str.contains('^Unnamed')]
            
            # Convert all values to string and handle NaN values
            for column in source_config_instance.columns:
                source_config_instance[column] = source_config_instance[column].apply(
                    lambda x: None if pd.isna(x) else str(x)
                )
            
            # Create Spark DataFrame with StringType schema for consistent typing
            schema = StructType([StructField(col_name, StringType(), True) for col_name in source_config_instance.columns])
            spark_source_config_instance = spark.createDataFrame(source_config_instance, schema=schema)
            
            # Add metadata timestamp column
            spark_source_config_instance_updated = spark_source_config_instance.withColumn('dataLoadedAt', date_format(current_timestamp(), "yyyy-MM-dd'T'HH:mm:ss"))
            
            # Write to Delta table with schema overwrite capability
            spark_source_config_instance_updated.write.mode('overwrite')\
                .option('overwriteSchema', 'true')\
                .format('delta')\
                .save(f'{lh_utils_path}/Tables/{target_schema_name}/{target_table_name}')
            
            print(f'Successfully loaded {sheet_name} -> {target_schema_name}.{target_table_name}')
            
        except Exception as e:
            print(f"Error processing {sheet_name} -> {target_schema_name}.{target_table_name}: {str(e)}")
            traceback.print_exc()

StatementMeta(, fbd9461f-47e2-4653-9407-cf8584579aee, 11, Finished, Available, Finished)

#### Main Function

In [10]:
# Read master_config
master_source_config = pd.read_excel(config_path,sheet_name = 'master_config')

# Select active sheets data
filtered_master_source_config = master_source_config[master_source_config['is_active'] == 1]

# Laod the active sheets into the LH
load_into_lakehouse(filtered_master_source_config)

StatementMeta(, fbd9461f-47e2-4653-9407-cf8584579aee, 12, Finished, Available, Finished)

Successfully loaded rule_binding -> dq_config.rule_binding
Successfully loaded data_entity -> dq_config.data_entity
Successfully loaded accepted_values -> dq_config.accepted_values
Successfully loaded formats -> dq_config.formats
Successfully loaded dataset -> dq_config.dataset


#### Stop the Session

In [ ]:
# Stop the Spark session
mssparkutils.session.stop()

StatementMeta(, , -1, Finished, , Finished)

SessionStateError: Livy session has failed. Session state: Killed. Session has been cancelled. Source: User.